In [5]:
"""
extract_uris.py
---------------
Scans all .html files in the Annotated folder, extracts every
<manual_label> and <auto_label> whose labelname is one of:
  - "legislation"
  - "decision"
  - "secondary sources"

For each match it records:
  - source_file  : the .html filename
  - labelname    : the labelname attribute
  - docid        : the docid attribute
  - uri          : the uri attribute
  - text         : the text content inside the tag

Results are saved as  uri_dataset.json  inside the same Annotated folder.

Usage
-----
    python extract_uris.py

Make sure beautifulsoup4 is installed:
    pip install beautifulsoup4
"""

import json
import os
from pathlib import Path
from bs4 import BeautifulSoup

# ── Configuration ────────────────────────────────────────────────────────────

ANNOTATED_DIR = Path(
    r"C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\final\Annotated"
)

TARGET_LABELNAMES = {"legislation", "decision", "secondary sources"}
TAG_NAMES = ["manual_label", "auto_label"]
OUTPUT_FILE = ANNOTATED_DIR / "uri_dataset.json"

# ── Main extraction ──────────────────────────────────────────────────────────

def extract_from_file(filepath: Path) -> list[dict]:
    """Return a list of record dicts for every matching label tag in *filepath*."""
    records = []
    with open(filepath, encoding="utf-8", errors="replace") as fh:
        soup = BeautifulSoup(fh, "html.parser")

    for tag_name in TAG_NAMES:
        for tag in soup.find_all(tag_name):
            labelname = tag.get("labelname", "")
            if labelname not in TARGET_LABELNAMES:
                continue

            records.append(
                {
                    "source_file": filepath.name,
                    "labelname":   labelname,
                    "docid":       tag.get("docid", None),
                    "uri":         tag.get("uri", None),
                    "text":        tag.get_text(strip=True),
                }
            )

    return records


def main():
    if not ANNOTATED_DIR.exists():
        raise FileNotFoundError(f"Directory not found: {ANNOTATED_DIR}")

    html_files = sorted(ANNOTATED_DIR.glob("*.html"))
    if not html_files:
        print("No .html files found in", ANNOTATED_DIR)
        return

    all_records: list[dict] = []

    for html_file in html_files:
        file_records = extract_from_file(html_file)
        all_records.extend(file_records)
        print(f"  {html_file.name:50s}  →  {len(file_records):4d} records")

    # Write output
    with open(OUTPUT_FILE, "w", encoding="utf-8") as out:
        json.dump(all_records, out, ensure_ascii=False, indent=2)

    print(f"\n✓ Extracted {len(all_records)} records from {len(html_files)} files.")
    print(f"✓ Saved to: {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

  1989CanLII1415CITT_annotated_GL_revRL_tech.html     →    94 records
  1994CanLII4528NLCA_annotated_GL_revRL.html          →   161 records
  1997CanLII16226_ONCA_annotated_EG_revRL_tech.html   →   623 records
  2001CanLII21117QCTDP_annotated_GL_revRL_tech.html   →   322 records
  2002SCC33_annotated_EG_revRL.html                   →   328 records
  2005QCCA437_LLMv1.3_Verified_GL_revRL_tech.html     →   111 records
  2008CSC9_annotated_EG_revRL.html                    →   484 records
  2016NBOMB12_annotated_EG_revRL.html                 →   132 records
  2019SCC65_annotated_EG_revRL_tech.html              →  1443 records
  2021QCCA1675_annotated_EG_revRL_tech.html           →   100 records
  2024NBKB203_annotated_VP_rev2RL_tech.html           →   244 records

✓ Extracted 4042 records from 11 files.
✓ Saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\final\Annotated\uri_dataset.json


In [7]:
"""
extract_unique_uris.py
----------------------
Reads uri_dataset.json (produced by extract_uris.py) and outputs:
  - A summary of unique URI counts (printed to console)
  - uri_list.json : a sorted list of unique URIs only

Usage
-----
    python extract_unique_uris.py

Both files must be in the same Annotated folder.
"""

import json
from pathlib import Path

# ── Configuration ─────────────────────────────────────────────────────────────

ANNOTATED_DIR = Path(
    r"C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\final\Annotated"
)

INPUT_FILE  = ANNOTATED_DIR / "mention_list.json"
OUTPUT_FILE = ANNOTATED_DIR / "uri_list.json"

# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    if not INPUT_FILE.exists():
        raise FileNotFoundError(f"Input file not found: {INPUT_FILE}\nRun extract_uris.py first.")

    with open(INPUT_FILE, encoding="utf-8") as fh:
        records = json.load(fh)

    all_uris   = [r["uri"] for r in records]
    total      = len(all_uris)

    # Separate None / "None" from real URIs
    none_values = {"None", "none", "", None}
    real_uris   = [u for u in all_uris if u not in none_values]
    unique_uris = sorted(set(real_uris))

    none_count   = total - len(real_uris)
    unique_count = len(unique_uris)

    # Console summary
    print(f"Total records          : {total}")
    print(f"Records with no URI    : {none_count}")
    print(f"Records with a URI     : {len(real_uris)}")
    print(f"Unique URIs            : {unique_count}")

    # Save
    with open(OUTPUT_FILE, "w", encoding="utf-8") as out:
        json.dump(unique_uris, out, ensure_ascii=False, indent=2)

    print(f"\n✓ Saved {unique_count} unique URIs to: {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

Total records          : 4042
Records with no URI    : 1772
Records with a URI     : 2270
Unique URIs            : 505

✓ Saved 505 unique URIs to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\final\Annotated\uri_list.json
